# 第18章：条件方差与预测时钟

先手算已实现方差和条件方差递推，再调用analyze_constructed生成360行构造路径，将constructed_simulation.csv写入RUN_DIR。French危机窗口及历史图默认跳过，因此本次不核验市场ZIP/CSV，也不生成四幅历史图。固定均值创新下的递推可用第16章知识理解；第17章的单位根与协整提供背景，不是这些公式的必需输入。

[本语言逐本运行模式与准备条件](../NOTEBOOK_GUIDE.md)。运行步骤、数据要求和预期输出见本指南。

In [ ]:
from pathlib import Path
import sys, os, uuid
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside the reader package tree")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch18_volatility_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
# French branches remain disabled until data rights and execution authorization are resolved.
RUN_FRENCH = False
print("CONSTRUCTED teaching route; French market branches SKIPPED; outputs:", RUN_DIR)

from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image, display

sys.path.insert(0, str(ROOT / "code"))

import ch18_volatility as workflow
from quant_course.volatility import (
    GARCHFit, GARCHParameters, ewma_variance_path, garch_variance_forecast,
    garch_variance_path, realized_variance,
)
print("Project root:", ROOT)

## 1. 先核手算：一段路径不是净收益平方

同一区间三个子区间的对数收益为 1%、−2%、3%。平方和是该网格的 RV；净对数收益平方只记录净位移。EWMA/ARCH/GARCH手算则把同样的三个数解释为逐月收益创新，目标不同。随后用共享核心核验 EWMA、ARCH/GARCH 和逐期多步预测。

In [ ]:
parts = [0.01, -0.02, 0.03]
rv = realized_variance(parts)
net_squared = sum(parts) ** 2
assert abs(rv - 0.0014) < 1e-12
assert abs(net_squared - 0.0004) < 1e-12
idx = pd.period_range("2000-01", periods=3, freq="M")
shocks = pd.Series(parts, index=idx)
params = GARCHParameters(omega=0.00001, alpha=0.1, beta=0.8)
ewma = ewma_variance_path(shocks, decay=0.9, initial_variance=0.0004)
garch = garch_variance_path(shocks, params, initial_variance=0.0004)
arch = garch_variance_path(shocks, GARCHParameters(0.00001, 0.1, 0.0), initial_variance=0.0004)
toy_fit = GARCHFit(params, 0.0, pd.Period("2000-03", freq="M"), 60, 0.0004, 0.0, 1, garch)
multi_step = garch_variance_forecast(toy_fit, steps=3)
assert abs(ewma.iloc[0] - 0.0004) < 1e-12
assert abs(ewma.iloc[1] - 0.00037) < 1e-12
assert abs(ewma.iloc[2] - 0.000373) < 1e-12
assert all(abs(actual - expected) < 1e-12 for actual, expected in zip(
    arch.tolist(), [0.0004, 0.00002, 0.00005]))
assert all(abs(actual - expected) < 1e-12 for actual, expected in zip(
    garch.tolist(), [0.0004, 0.00034, 0.000322]))
print({"RV": rv, "net_return_squared": net_squared,
       "EWMA h1:h3": ewma.tolist(), "ARCH h1:h3": arch.tolist(),
       "GARCH h1:h3": garch.tolist(),
       "GARCH long-run h": params.unconditional_variance,
       "expected one-month h for horizons 1:3": multi_step.tolist()})

**读数**：RV 与净对数收益平方差别来自方向抵消。EWMA/GARCH 返回同一单位的逐期方差路径；将其开平方才得到小数收益标准差。共享核心的输入创新在每期结束后才用于更新后一期。

In [ ]:
summary, simulation = workflow.analyze_constructed()
simulation.to_csv(RUN_DIR / "constructed_simulation.csv")
print(summary)
assert summary["simulation"]["observations"] == 360


## 2. 构造输出与跳过的历史窗口

上格新算360项观察的构造路径并写CSV。下格French分支禁用，默认不核市场ZIP/CSV，也不拟合204个训练月。1990–2006年训练和2007–2009年36个月评价只属于历史方案；目标月平方创新在月后才可得。

In [ ]:
if RUN_FRENCH:
    actual = pd.read_csv(ROOT / "outputs/ch18/ch18_actual_forecasts.csv", parse_dates=False)
    actual.head(), actual.tail(), len(actual)
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


In [ ]:
print("SKIPPED historical figure gallery; constructed calculations above are freshly computed; French historical images are not reproduced")


## 3. 输出的含义与边界

构造路径有已知true_variance，可比较GARCH生成机制、预测与嘈杂平方创新代理。有限burn-in只减弱初始化影响，不保证精确平稳抽样。这是教学代码验证，不是市场证据。

若阅读被跳过的French历史报告，QLIKE最低代理损失不代表交易冠军。真实序列没有潜在逐月条件方差标签，均值误设可能使平方代理产生条件偏差。36个月危机切片不支持胜者显著性。虽递推预测只用此前回报，缺少历史发布/修订版本仍不能称PIT或当时可执行OOS。没有仓位或成交账本，成本后收益不适用，不能记为成本为零。

In [ ]:
simulation_summary = summary["simulation"]
simulation_summary